# 12 — Further Clustering Experiments

**Experiment D — RoBERTa + tuned UMAP:** Apply the same 3×3 UMAP grid
(n_neighbors ∈ {15,30,50} × min_dist ∈ {0.0,0.05,0.1}) to RoBERTa
(768-dim) that was applied to MiniLM in notebook 11 Experiment B.
RoBERTa with default UMAP scored only ~24% ACC; tuning should close the gap.

**Experiment E — Combined embeddings:** L2-normalise MiniLM summary
embeddings and MiniLM title embeddings, concatenate → (3824, 768),
apply UMAP, cluster.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
import umap
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

from utils import config
from utils.interpretability import summarize_clusters
from utils.metrics import evaluate_unsupervised, hungarian_match_predictions
from utils.samples import save_full_output

train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
true_labels  = train_clean["label"].to_numpy()
texts        = train_clean["text"].tolist()
summaries    = train_clean["summary"].tolist()

assert len(true_labels) == 3824, f"Expected 3824 rows, got {len(true_labels)}"
print(f"Rows: {len(train_clean)} | k={config.NUM_CLASSES}")

all_results = {}
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

Rows: 3824 | k=16


In [2]:
def umap_reduce(
    emb: np.ndarray,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    n_components: int = 50,
    metric: str = "cosine",
    seed: int = config.SEED,
) -> np.ndarray:
    reducer = umap.UMAP(
        n_components=n_components,
        metric=metric,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        random_state=seed,
    )
    return reducer.fit_transform(emb)


def run_and_save(name: str, cluster_labels: np.ndarray, emb_original: np.ndarray) -> None:
    metrics = evaluate_unsupervised(
        true_labels, cluster_labels, emb_original,
        metric_sample_size=config.SILHOUETTE_SAMPLE_SIZE, seed=config.SEED,
    )
    all_results[name] = metrics

    with open(config.RESULTS_DIR / f"metrics_{name}.json", "w") as f:
        json.dump(metrics, f, indent=2)

    np.save(config.RESULTS_DIR / f"cluster_labels_{name}.npy", cluster_labels)

    interp = summarize_clusters(
        summaries, cluster_labels, true_labels,
        emb_original,
        config.CLASS_NAMES,
    )
    interp.to_csv(config.RESULTS_DIR / f"clusters_{name}.csv", index=False)

    predicted = hungarian_match_predictions(true_labels, cluster_labels)
    save_full_output(
        texts, predicted, true_labels, config.CLASS_NAMES,
        extra_columns={"summary": summaries},
        path=config.RESULTS_DIR / f"full_labels_{name}.csv",
    )

    print(f"  {name}: ACC={metrics['ACC (Hungarian)']:.4f}  "
          f"F1={metrics['Macro F1']:.4f}  "
          f"NMI={metrics['NMI']:.4f}")


print("Helpers defined.")

Helpers defined.


In [3]:
# ════════════════════════════════════════════════════════════════════
# EXPERIMENT D — RoBERTa + tuned UMAP
# ════════════════════════════════════════════════════════════════════
roberta_emb = np.load(
    REPO_ROOT / "embeddings_cache" / "roberta_train_full_summary.npy"
).astype(np.float32)
assert roberta_emb.shape == (3824, 768), f"Unexpected: {roberta_emb.shape}"
print(f"RoBERTa embeddings loaded: {roberta_emb.shape}")

N_NEIGHBORS_GRID = [15, 30, 50]
MIN_DIST_GRID    = [0.0, 0.05, 0.1]

print("\nUMAP grid search on RoBERTa (3x3 = 9 combos, scored by KMeans silhouette):")
print(f"{'n_neighbors':>12} {'min_dist':>9} {'silhouette':>12}")
print("-" * 36)

best_score, best_n_neighbors, best_min_dist = -1.0, 15, 0.1

for nn in N_NEIGHBORS_GRID:
    for md in MIN_DIST_GRID:
        reduced = umap_reduce(roberta_emb, n_neighbors=nn, min_dist=md)
        km_tmp = KMeans(n_clusters=config.NUM_CLASSES, random_state=config.SEED, n_init=10)
        labels_tmp = km_tmp.fit_predict(reduced)
        score = silhouette_score(
            reduced, labels_tmp,
            sample_size=config.SILHOUETTE_SAMPLE_SIZE,
            random_state=config.SEED,
        )
        marker = " <- best" if score > best_score else ""
        print(f"{nn:>12} {md:>9.2f} {score:>12.4f}{marker}")
        if score > best_score:
            best_score, best_n_neighbors, best_min_dist = score, nn, md

print(f"\nBest params: n_neighbors={best_n_neighbors}, min_dist={best_min_dist} "
      f"(silhouette={best_score:.4f})")

RoBERTa embeddings loaded: (3824, 768)

UMAP grid search on RoBERTa (3x3 = 9 combos, scored by KMeans silhouette):
 n_neighbors  min_dist   silhouette
------------------------------------


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


          15      0.00       0.4409 <- best


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


          15      0.05       0.4079


...

          50      0.00       0.4250


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


          50      0.05       0.4020


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


          50      0.10       0.3705

Best params: n_neighbors=15, min_dist=0.0 (silhouette=0.4409)


In [4]:
print("\n=== [D] RoBERTa KMeans (tuned UMAP) ===")
roberta_reduced_tuned = umap_reduce(
    roberta_emb, n_neighbors=best_n_neighbors, min_dist=best_min_dist
)
print(f"  UMAP reduced: {roberta_reduced_tuned.shape}")

km = KMeans(n_clusters=config.NUM_CLASSES, random_state=config.SEED, n_init=10)
run_and_save("roberta_kmeans_umap", km.fit_predict(roberta_reduced_tuned), roberta_emb)

print("\n=== [D] RoBERTa Agglomerative (tuned UMAP) ===")
agg = AgglomerativeClustering(
    n_clusters=config.NUM_CLASSES,
    linkage="ward",
    metric="euclidean",
)
run_and_save("roberta_agglomerative_umap", agg.fit_predict(roberta_reduced_tuned), roberta_emb)


=== [D] RoBERTa KMeans (tuned UMAP) ===


C:\Users\ACER\OneDrive\Documents\final-project\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  UMAP reduced: (3824, 50)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  roberta_kmeans_umap: ACC=0.2482  F1=0.2392  NMI=0.2522

=== [D] RoBERTa Agglomerative (tuned UMAP) ===


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  roberta_agglomerative_umap: ACC=0.2547  F1=0.2493  NMI=0.2342


In [5]:
import json
for name in ["roberta_kmeans_umap", "roberta_agglomerative_umap"]:
    m = json.load(open(config.RESULTS_DIR / f"metrics_{name}.json"))
    assert m["ACC (Hungarian)"] > 0.24, f"{name} did not improve over baseline"
    print(f"{name}: ACC={m['ACC (Hungarian)']:.4f}")

roberta_kmeans_umap: ACC=0.2482
roberta_agglomerative_umap: ACC=0.2547
